# Rerank ViDoRe v3 HR text (full dataset)

This notebook compares **No Reranker** (BM25 candidate order) with **Llama Nemotron Rerank VL** on the same post-retrieval text. It uses **all pages, queries, and relevance judgments in [ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr)**—one dataset within the larger ViDoRe v3 collection, not the entire collection. The dataset card lists 1,110 pages and 1,908 queries. For a simple, fast starting point, use the [synthetic text-chunk notebook](reranking_existing_retrieval_results_text_only.ipynb).

**Plan for a long run.** With 12 candidates per query, the full HR evaluation scores about 22,896 query–page pairs. Hosted mode makes about 1,908 API requests and may incur charges or rate limits; the notebook asks for confirmation before sending them. This is a **text-only worked comparison**, not an official visual ViDoRe benchmark score: it uses BM25 candidates and capped extracted markdown, not page images. Reranking cannot recover a relevant page absent from the candidate list.

## Configuration and dependencies

Set INFERENCE_MODE to "hosted" (the default) or "local". Both paths send **only extracted page markdown** to the same VL reranker; neither sends page images to the model. Hosted mode requires NVIDIA_API_KEY but no local GPU. Local mode needs a CUDA GPU and the model weights. Loading the full HR corpus still downloads roughly 442 MB of source data on the first run because the dataset stores images in its corpus files, even though this notebook does not decode or rerank them. Allow time and disk space for the download.

The optional Jupyter magic below installs into the selected kernel. Uncomment only what your chosen mode needs, then restart the kernel. Local inference uses the model card's direct Transformers processor path with PyTorch SDPA, so FlashAttention and Sentence Transformers are not required. Install a CUDA-enabled PyTorch build matched to your driver using the [PyTorch selector](https://pytorch.org/get-started/locally/); do not install a generic CPU wheel if you intend to run locally. The dataset card identifies the pinned revision used for end-to-end evaluation.

In [1]:
# %pip install --upgrade "datasets>=4.8.2,<5"
# Local mode only, after installing a matching CUDA-enabled PyTorch build:
# %pip install --upgrade "transformers>=4.56.0"

from importlib.util import find_spec

missing = [name for name in ("datasets",) if find_spec(name) is None]
if missing:
    raise ImportError(
        f"Missing {', '.join(missing)} in this kernel. Run the optional %pip command above, "
        "restart the kernel, and run from the top."
    )

DATASET_ID = "vidore/vidore_v3_hr"
DATASET_REVISION = "95f2f83a5a09590a89e34960479f9438e48bca77"
INFERENCE_MODE = "local"  # "hosted" or "local"
CANDIDATES_PER_QUERY = 12
TOP_K = 5
DISPLAY_QUERY_ROWS = 10
MAX_PASSAGE_CHARS = 4000
RERANK_BATCH_SIZE = 8
RERANK_MAX_LENGTH = 512
RERANK_MODEL = "nvidia/llama-nemotron-rerank-vl-1b-v2"
RERANK_URL = (
    "https://ai.api.nvidia.com/v1/retrieval/nvidia/"
    "llama-nemotron-rerank-vl-1b-v2/reranking"
)
assert INFERENCE_MODE in {"hosted", "local"}
assert CANDIDATES_PER_QUERY >= TOP_K > 0
assert DISPLAY_QUERY_ROWS > 0 and MAX_PASSAGE_CHARS > 0
assert RERANK_BATCH_SIZE > 0 and RERANK_MAX_LENGTH > 0

if INFERENCE_MODE == "local":
    local_missing = [name for name in ("torch", "transformers")
                     if find_spec(name) is None]
    if local_missing:
        raise ImportError("Local mode requires " + ", ".join(local_missing) +
                          ". Use the optional local install guidance above.")
    from importlib.metadata import version
    from packaging.version import Version
    if Version(version("transformers")) < Version("4.56.0"):
        raise RuntimeError("Local VL reranking requires transformers>=4.56.0.")
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Local mode needs a CUDA GPU visible to this kernel.")

## Load all HR pages, queries, and judgments

ViDoRe v3 HR supplies page images, extracted markdown, query text, and graded qrels (1 = critically relevant, 2 = fully relevant). We load the entire HR test split but select only corpus IDs and markdown for BM25 and reranking, so no page images are decoded. Unjudged pages are treated as relevance 0. This still evaluates only HR, not the other ViDoRe v3 datasets.

In [2]:
from datasets import load_dataset

def load_split(config):
    return load_dataset(
        DATASET_ID, config, split="test", revision=DATASET_REVISION,
    )

corpus_text = load_split("corpus").select_columns(["corpus_id", "markdown"])
pages = [
    {"corpus_id": row["corpus_id"], "markdown": row["markdown"] or ""}
    for row in corpus_text
]
queries = sorted(load_split("queries"), key=lambda row: row["query_id"])
page_by_id = {row["corpus_id"]: row for row in pages}
relevance_by_query = {}
for row in load_split("qrels"):
    if row["corpus_id"] not in page_by_id:
        raise ValueError("A qrel refers to a page outside the HR corpus.")
    judgments = relevance_by_query.setdefault(row["query_id"], {})
    page_id = row["corpus_id"]
    judgments[page_id] = max(judgments.get(page_id, 0), row["score"])

print(f"Loaded all {len(pages)} HR pages, {len(queries)} queries, and "
      f"judgments for {len(relevance_by_query)} queries.")
if len(pages) < CANDIDATES_PER_QUERY:
    raise ValueError("CANDIDATES_PER_QUERY exceeds the HR corpus size.")
if any(row["query_id"] not in relevance_by_query for row in queries):
    raise ValueError("Some HR queries have no relevance judgments.")

/home/edwardk/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded all 1110 HR pages, 1908 queries, and judgments for 1908 queries.


## Produce post-retrieval candidates

BM25 is the first-stage retriever here. It searches **every HR page for every HR query**, then returns the top CANDIDATES_PER_QUERY pages to either option. In an application, replace its output with chunks or pages from your existing retriever and keep the reranking cells unchanged. Queries with no relevant page among those candidates remain in the evaluation: a reranker cannot fix that retrieval miss.

In [3]:
import math
import re
from collections import Counter

def tokens(text):
    return re.findall(r"[\w]+", text.lower())

page_tokens = [tokens(row["markdown"]) for row in pages]
term_counts = [Counter(row) for row in page_tokens]
document_frequency = Counter(term for row in term_counts for term in row)
average_length = sum(map(len, page_tokens)) / len(page_tokens)

def retrieve(query, depth=CANDIDATES_PER_QUERY):
    query_terms = set(tokens(query))
    scores = []
    for row, counts, words in zip(pages, term_counts, page_tokens):
        score = 0.0
        for term in query_terms:
            frequency = counts[term]
            if not frequency:
                continue
            df = document_frequency[term]
            idf = math.log(1 + (len(pages) - df + 0.5) / (df + 0.5))
            score += idf * frequency * 2.2 / (
                frequency + 1.2 * (0.25 + 0.75 * len(words) / average_length)
            )
        scores.append((score, row["corpus_id"]))
    best = sorted(scores, key=lambda pair: (-pair[0], pair[1]))[:depth]
    return [
        {"corpus_id": page_id, "retrieval_score": score, "original_rank": rank}
        for rank, (score, page_id) in enumerate(best, start=1)
    ]

examples = [
    {"query_id": row["query_id"], "query": row["query"],
     "candidates": retrieve(row["query"]),
     "relevance": relevance_by_query[row["query_id"]]}
    for row in queries
]
zero_recall = sum(
    not any(hit["corpus_id"] in example["relevance"]
            for hit in example["candidates"])
    for example in examples
)
print(f"Prepared {len(examples)} HR queries with "
      f"{CANDIDATES_PER_QUERY} BM25 candidates each.")
print(f"{zero_recall} queries have no judged-relevant page in their candidate set.")

Prepared 1908 HR queries with 12 BM25 candidates each.
1331 queries have no judged-relevant page in their candidate set.


## Rerank the same candidates with Llama Nemotron Rerank VL

Both modes rerank the **same text-only candidates**. Hosted mode calls the [VL reranking API](https://docs.api.nvidia.com/nim/reference/nvidia-llama-nemotron-rerank-vl-1b-v2-infer) with text passages. Local mode follows the [model card's Transformers recipe](https://huggingface.co/nvidia/llama-nemotron-rerank-vl-1b-v2): each pair has a question, doc_text, and an empty doc_image. Extracted markdown is capped to keep inputs manageable; changing that cap or the local token limit can change results. No page images are sent or decoded by either mode.

For hosted mode, export NVIDIA_API_KEY before starting Jupyter or enter it in the hidden prompt. Hosted mode sends all HR queries and candidate markdown to NVIDIA; local mode keeps them on your machine but loads the model onto a CUDA GPU. The next cell asks for confirmation before the full run. An interrupted run can resume within the same kernel without repeating completed queries.

In [4]:
import json
import os
import time
from getpass import getpass
from urllib.error import HTTPError
from urllib.request import Request, urlopen

def require_api_key():
    key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if not key:
        key = getpass("NVIDIA API key: ").strip()
    if not key:
        raise EnvironmentError("NVIDIA_API_KEY is required for hosted reranking.")
    os.environ["NVIDIA_API_KEY"] = key
    return key

local_processor = None
local_reranker = None
if INFERENCE_MODE == "local":
    from transformers import AutoModelForSequenceClassification, AutoProcessor

    local_processor = AutoProcessor.from_pretrained(
        RERANK_MODEL, trust_remote_code=True, rerank_max_length=RERANK_MAX_LENGTH
    )
    rerank_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    local_reranker = AutoModelForSequenceClassification.from_pretrained(
        RERANK_MODEL, trust_remote_code=True, torch_dtype=rerank_dtype,
        attn_implementation="sdpa",
    ).eval().to("cuda")

def local_rerank(query, candidates):
    scores = []
    for start in range(0, len(candidates), RERANK_BATCH_SIZE):
        batch = candidates[start : start + RERANK_BATCH_SIZE]
        pairs = [
            {"question": query,
             "doc_text": page_by_id[hit["corpus_id"]]["markdown"][:MAX_PASSAGE_CHARS] or " ",
             "doc_image": ""}
            for hit in batch
        ]
        inputs = local_processor.process_queries_documents_crossencoder(pairs)
        inputs = {
            name: value.to("cuda") if isinstance(value, torch.Tensor) else value
            for name, value in inputs.items()
        }
        with torch.inference_mode():
            logits = local_reranker(**inputs, return_dict=True).logits
        scores.extend(float(score) for score in logits.view(-1).cpu().tolist())
    if len(scores) != len(candidates) or any(not math.isfinite(score) for score in scores):
        raise ValueError("Local reranker must return one finite score per candidate.")
    reranked = [dict(hit, rerank_score=score) for hit, score in zip(candidates, scores)]
    return sorted(reranked, key=lambda hit: hit["rerank_score"], reverse=True)

def hosted_rerank(query, candidates):
    passages = [
        {"text": page_by_id[hit["corpus_id"]]["markdown"][:MAX_PASSAGE_CHARS] or " "}
        for hit in candidates
    ]
    request = Request(
        RERANK_URL,
        data=json.dumps({"model": RERANK_MODEL, "query": {"text": query},
                         "passages": passages, "truncate": "END"}).encode("utf-8"),
        headers={"Authorization": f"Bearer {require_api_key()}",
                 "Accept": "application/json", "Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=180) as response:
            result = json.load(response)
    except HTTPError as exc:
        raise RuntimeError(f"VL reranker returned HTTP {exc.code}; check API access and input limits.") from exc
    scores = [None] * len(candidates)
    for item in result["rankings"]:
        index = int(item["index"])
        if not 0 <= index < len(scores) or scores[index] is not None:
            raise ValueError("Invalid candidate index in reranker response.")
        scores[index] = float(item["logit"])
    if any(score is None or not math.isfinite(score) for score in scores):
        raise ValueError("Reranker must return one finite score per candidate.")
    reranked = [dict(hit, rerank_score=score) for hit, score in zip(candidates, scores)]
    return sorted(reranked, key=lambda hit: hit["rerank_score"], reverse=True)

def rerank_candidates(query, candidates):
    if INFERENCE_MODE == "local":
        return local_rerank(query, candidates)
    return hosted_rerank(query, candidates)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████████████| 601/601 [00:00<00:00, 3474.25it/s]


In [5]:
pending = [example for example in examples if "reranked" not in example]
print(f"Pending: {len(pending)} {INFERENCE_MODE} reranking queries, up to "
      f"{len(pending) * CANDIDATES_PER_QUERY} query–page pairs.")
if pending:
    confirmation = input("Full HR reranking may take a long time; hosted mode may incur API charges. Type RUN to continue: ")
    if confirmation.strip() != "RUN":
        raise RuntimeError("Full HR reranking cancelled before inference.")
    if INFERENCE_MODE == "hosted":
        require_api_key()
    for completed, example in enumerate(pending, start=1):
        started = time.perf_counter()
        example["reranked"] = rerank_candidates(example["query"], example["candidates"])
        example["rerank_seconds"] = time.perf_counter() - started
        if completed % 100 == 0 or completed == len(pending):
            print(f"Completed {completed}/{len(pending)} pending queries.")
print(f"Reranked {len(examples)} HR queries with {RERANK_MODEL} ({INFERENCE_MODE}).")

Pending: 1908 local reranking queries, up to 22896 query–page pairs.


Full HR reranking may take a long time; hosted mode may incur API charges. Type RUN to continue:  RUN


Completed 100/1908 pending queries.
Completed 200/1908 pending queries.
Completed 300/1908 pending queries.
Completed 400/1908 pending queries.
Completed 500/1908 pending queries.
Completed 600/1908 pending queries.
Completed 700/1908 pending queries.
Completed 800/1908 pending queries.
Completed 900/1908 pending queries.
Completed 1000/1908 pending queries.
Completed 1100/1908 pending queries.
Completed 1200/1908 pending queries.
Completed 1300/1908 pending queries.
Completed 1400/1908 pending queries.
Completed 1500/1908 pending queries.
Completed 1600/1908 pending queries.
Completed 1700/1908 pending queries.
Completed 1800/1908 pending queries.
Completed 1900/1908 pending queries.
Completed 1908/1908 pending queries.
Reranked 1908 HR queries with nvidia/llama-nemotron-rerank-vl-1b-v2 (local).


## Measure the difference

nDCG@K uses ViDoRe's graded qrels and rewards relevant pages near the top. Its ideal ordering uses **all judged HR pages**, including relevant pages BM25 failed to retrieve. Candidate recall is unchanged by reranking. The notebook shows only the first DISPLAY_QUERY_ROWS query rows plus full-HR means to keep output readable. Rerank time includes model processing and, in hosted mode, network time; it is not a controlled latency benchmark.

In [6]:
from IPython.display import Markdown, display

def ndcg_at_k(ranking, judgments, k):
    def dcg(scores):
        return sum((2**score - 1) / math.log2(rank + 1)
                   for rank, score in enumerate(scores, start=1))
    observed = [judgments.get(hit["corpus_id"], 0) for hit in ranking[:k]]
    ideal = sorted(judgments.values(), reverse=True)[:k]
    denominator = dcg(ideal)
    return dcg(observed) / denominator if denominator else 0.0

def table(rows, columns):
    def value(item):
        return (f"{item:.3f}" if isinstance(item, float) else str(item)).replace("|", "\\|")
    lines = ["| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(value(row[col]) for col in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

results = []
for example in examples:
    before = ndcg_at_k(example["candidates"], example["relevance"], TOP_K)
    after = ndcg_at_k(example["reranked"], example["relevance"], TOP_K)
    relevant_ids = set(example["relevance"])
    retrieved_ids = {hit["corpus_id"] for hit in example["candidates"]}
    results.append({"query_id": example["query_id"],
                    "No Reranker nDCG": before, "VL Reranker nDCG": after,
                    "delta": after - before,
                    "candidate recall": len(retrieved_ids & relevant_ids) / len(relevant_ids),
                    "rerank seconds": example["rerank_seconds"]})
metric_columns = ["query_id", "No Reranker nDCG", "VL Reranker nDCG",
                  "delta", "candidate recall", "rerank seconds"]
table(results[:DISPLAY_QUERY_ROWS], metric_columns)
print(f"Showing {min(DISPLAY_QUERY_ROWS, len(results))} of {len(results)} HR queries above.")
table([{"option": option, f"mean nDCG@{TOP_K}": sum(row[key] for row in results) / len(results)}
       for option, key in [("No Reranker", "No Reranker nDCG"),
                           ("Llama Nemotron Rerank VL", "VL Reranker nDCG")]],
      ["option", f"mean nDCG@{TOP_K}"])

| query_id | No Reranker nDCG | VL Reranker nDCG | delta | candidate recall | rerank seconds |
| --- | --- | --- | --- | --- | --- |
| 0 | 0.000 | 0.000 | 0.000 | 0.000 | 0.567 |
| 1 | 0.000 | 0.146 | 0.146 | 0.071 | 0.195 |
| 2 | 0.316 | 0.684 | 0.369 | 0.500 | 0.194 |
| 3 | 0.146 | 0.131 | -0.015 | 0.167 | 0.192 |
| 4 | 0.307 | 0.501 | 0.195 | 1.000 | 0.195 |
| 5 | 0.642 | 0.478 | -0.165 | 0.750 | 0.201 |
| 6 | 1.000 | 1.000 | 0.000 | 1.000 | 0.200 |
| 7 | 1.000 | 1.000 | 0.000 | 1.000 | 0.191 |
| 8 | 0.237 | 0.613 | 0.376 | 0.500 | 0.199 |
| 9 | 0.854 | 0.786 | -0.068 | 0.429 | 0.193 |

Showing 10 of 1908 HR queries above.


| option | mean nDCG@5 |
| --- | --- |
| No Reranker | 0.116 |
| Llama Nemotron Rerank VL | 0.187 |

In [7]:
first = examples[0]
comparison = [
    {"reranked_rank": rank, "BM25_rank": hit["original_rank"],
     "corpus_id": hit["corpus_id"],
     "relevance": first["relevance"].get(hit["corpus_id"], 0),
     "BM25_score": hit["retrieval_score"],
     "rerank_logit": hit["rerank_score"]}
    for rank, hit in enumerate(first["reranked"], start=1)
]
print(f"Detailed rank changes for query_id={first['query_id']}: {first['query']}")
table(comparison, list(comparison[0]))

Detailed rank changes for query_id=0: What impact did the shifts in temporary contract usage from 2018 to 2023 have on the difference in employment stability between mobile EU workers and citizens of the host countries?


| reranked_rank | BM25_rank | corpus_id | relevance | BM25_score | rerank_logit |
| --- | --- | --- | --- | --- | --- |
| 1 | 5 | 475 | 0 | 19.716 | -5.145 |
| 2 | 4 | 365 | 0 | 20.012 | -5.234 |
| 3 | 10 | 523 | 0 | 18.644 | -6.535 |
| 4 | 11 | 503 | 0 | 18.538 | -6.548 |
| 5 | 3 | 443 | 0 | 20.068 | -6.766 |
| 6 | 6 | 755 | 0 | 19.261 | -6.972 |
| 7 | 2 | 756 | 0 | 21.652 | -7.637 |
| 8 | 9 | 961 | 0 | 18.787 | -8.539 |
| 9 | 7 | 759 | 0 | 19.194 | -8.577 |
| 10 | 12 | 931 | 0 | 18.354 | -8.823 |
| 11 | 1 | 185 | 0 | 22.090 | -9.374 |
| 12 | 8 | 244 | 0 | 18.992 | -9.469 |

## Aggregate result across all HR queries

The final score is the mean nDCG across **all** HR queries, including queries with no judged-relevant page among the BM25 candidates. The delta is reranked mean minus no-reranker mean; a positive value means the reranker improved this text-only pipeline's aggregate ranking quality.

In [8]:
if len(results) != len(queries):
    raise ValueError("Aggregate results must include every HR query.")

mean_before = sum(row["No Reranker nDCG"] for row in results) / len(results)
mean_after = sum(row["VL Reranker nDCG"] for row in results) / len(results)
aggregate = {
    "queries": len(results),
    f"No Reranker mean nDCG@{TOP_K}": mean_before,
    f"VL Reranker mean nDCG@{TOP_K}": mean_after,
    f"delta mean nDCG@{TOP_K}": mean_after - mean_before,
}
table([aggregate], list(aggregate))

| queries | No Reranker mean nDCG@5 | VL Reranker mean nDCG@5 | delta mean nDCG@5 |
| --- | --- | --- | --- |
| 1908 | 0.116 | 0.187 | 0.071 |

## Interpret the result

- **Fixed candidates:** Both options use identical BM25 results. Reranking can change order, not recall. Increase the first-stage retrieval depth if positives are absent.
- **Quality versus cost:** More candidates can improve opportunities for reranking but increase inference and latency. This notebook evaluates every HR query, but its BM25 retrieval and capped model inputs differ from official benchmark pipelines.
- **Text-only limitation:** Both modes use extracted markdown, not page images. Charts, layout, or visual evidence absent from that text are unavailable to both BM25 and the reranker, even though the VL model itself supports images. These results are not a visual-document benchmark score.
- **Operational fit:** Hosted inference avoids a local GPU but sends document text to an endpoint and requires an API key and network access. Local inference keeps text in your environment but needs model storage and GPU memory. Check source-document licenses before redistributing outputs.
- **Other models:** A Nemotron 3.5 Rerank 8B option is not included because a public model ID and inference contract have not been verified. Do not interpret its absence as a quality comparison.